# Ejemplo de uso de scraper-x

Este cuaderno recorre el flujo completo con descargas pequeñas: comprobar el
entorno, verificar la sesión de X, descargar, revisar la calidad del resultado y
hacer un primer análisis.

**Antes de empezar**

1. Instala las dependencias del scraper y las del análisis:

   ```
   pip install -r requirements.txt
   pip install pandas matplotlib
   ```

2. Inicia sesión en X a mano en un perfil de Firefox y **cierra Firefox**
   (sección 3 del README). El scraper no inicia sesión por ti.

3. Abre este cuaderno **desde la carpeta de scraper-x**, para que encuentre los
   scripts y el censo de ejemplo.

Funciona igual en macOS, Linux y Windows: todo se ejecuta con el mismo intérprete
que corre el cuaderno.

## 1. Comprobar el entorno

Esta celda no abre el navegador. Sólo confirma que el scraper puede encontrar el
perfil de Firefox y el driver.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import selenium
import scraper_x

print("Python  ", sys.version.split()[0])
print("selenium", selenium.__version__)

perfil = scraper_x.discover_firefox_profile(None)
driver_path = scraper_x.resolve_driver_path(None)

print("perfil de Firefox:", perfil or "NO DETECTADO -> define X_FIREFOX_PROFILE")
print("geckodriver      :", driver_path or "lo descargará Selenium Manager")

## 2. Verificar la sesión de X

Esta celda **sí abre Firefox** (sin ventana) y mira si la búsqueda avanzada
responde. Firefox no puede estar abierto con el mismo perfil.

- `results` → sesión activa, se puede descargar.
- `login` → hay que entrar a X a mano en ese perfil.
- `error` o `False` → X está bloqueando o la página no cargó; espera y repite.

In [ ]:
import time

driver = scraper_x.create_driver(perfil, driver_path, True, 25.0)
try:
    driver.get(scraper_x.build_search_url("from:petrogustavo"))
    time.sleep(5)
    estado = scraper_x.page_state(driver)
finally:
    driver.quit()

print("estado de la página:", estado)
assert estado == "results", "Sin sesión activa: revisa la sección 3 del README."
print("Sesión lista.")

## 3. Ejecutar los scripts desde el cuaderno

Los scripts son programas de línea de comandos. Esta función los llama con el
mismo Python del cuaderno, así que no hay que preocuparse por `python` frente a
`py` en Windows.

In [ ]:
import subprocess

def correr(*argumentos, mostrar=4000):
    """Ejecuta uno de los scripts y muestra el final de su salida."""
    proceso = subprocess.run(
        [sys.executable, "-u", *argumentos],
        capture_output=True,
        text=True,
    )
    salida = (proceso.stdout or "")[-mostrar:]
    print(salida)
    if proceso.returncode:
        print("--- error ---")
        print((proceso.stderr or "")[-2000:])
    return proceso.returncode

correr("descargar_cuentas.py", "--roster", "ejemplo_pacto_historico.json",
       "--window-months", "3", "--plan-only")

## 4. Descarga por términos (ejemplo corto)

Dos días de un hashtag. Es la forma más rápida de ver el flujo completo: suele
tardar entre uno y tres minutos.

`--end` es exclusivo, así que `2026-06-22` cubre el 20 y el 21 de junio.

In [ ]:
correr("descargar_busqueda.py",
       "--term", "#PrioridadNacional",
       "--term", "Prioridad Nacional",
       "--start", "2026-06-20", "--end", "2026-06-22",
       "--batch-days", "1",
       "--retries", "4", "--retry-backoff", "120",
       "--output", "ejemplo_terminos.csv")

## 5. Descarga por cuentas (ejemplo corto)

Una sola cuenta del censo de ejemplo, un mes. Para tu propio estudio, cambia
`--roster` por tu censo y quita `--account` para bajar todas las cuentas.

Si X bloquea, el script espera y reintenta; puedes interrumpir la celda y
volver a ejecutarla, porque continúa donde quedó.

In [ ]:
correr("descargar_cuentas.py",
       "--roster", "ejemplo_pacto_historico.json",
       "--account", "senado-ivan-cepeda",
       "--start", "2026-06-01T00:00:00-05:00",
       "--end", "2026-07-01T00:00:00-05:00",
       "--window-months", "1",
       "--retries", "6", "--retry-backoff", "300",
       "--output", "ejemplo_cuenta.csv")

## 6. Cargar el resultado

El CSV usa `;` como separador y UTF-8 con BOM.

In [ ]:
import pandas as pd

ARCHIVO = "ejemplo_cuenta.csv"   # o "ejemplo_terminos.csv"

df = pd.read_csv(ARCHIVO, sep=";", encoding="utf-8-sig")
df["date"] = pd.to_datetime(df["date"], format="ISO8601", utc=True)

print(df.shape)
print(list(df.columns))
df[["date", "queried_handle", "favorites", "text"]].head(3)

## 7. Revisar la calidad antes de analizar

Tres comprobaciones que conviene hacer siempre. La tercera es la menos obvia: X
recorta las publicaciones largas a unos 280 caracteres **sin dejar ninguna
marca**, y eso se detecta por un pico de longitudes entre 270 y 285.

In [ ]:
largo = df["text"].str.len()

print("publicaciones        :", len(df))
print("ids duplicados       :", int(df["tweet_id"].duplicated().sum()))
print("textos vacíos        :", int(df["text"].str.strip().eq("").sum()))
if "queried_handle" in df:
    distintos = (df["username"].str.lower() != df["queried_handle"].str.lower()).sum()
    print("autor != cuenta      :", int(distintos))

banda = int(largo.between(270, 285).sum())
largas = int((largo > 300).sum())
print()
print("longitud: media %.0f | máx %d" % (largo.mean(), largo.max()))
print("en banda 270-285     : %d (%.1f%%)" % (banda, 100 * banda / len(df)))
print("con más de 300       :", largas)

# X recorta las publicaciones largas cerca de los 280 caracteres sin dejar marca.
# La señal no es el tamaño de la banda por sí sola, sino que la banda acumule
# muchas más publicaciones que el tramo de textos largos: si de verdad hubiera
# publicaciones extensas, estarían por encima de 300 en vez de apiladas en 280.
sospecha = 100 * banda / len(df) > 20 and banda > 3 * largas
print()
if len(df) < 300:
    print("Muestra pequeña: la comprobación de truncamiento necesita unos cientos "
          "de publicaciones para ser fiable.")
elif sospecha:
    print("SOSPECHA de textos cortados: reléelos con --force --refresh")
else:
    print("Sin señales de truncamiento sistemático.")


### Huecos por cuenta

Un silencio largo puede ser real (receso, cuenta inactiva) o ser un tramo que X
nunca entregó. La regla práctica: compara el hueco con el ritmo de publicación de
esa misma cuenta.

In [ ]:
def huecos(df, dias_minimos=14, esperadas_minimas=20):
    """Lista los silencios largos de cada cuenta según su propio ritmo."""
    filas = []
    for cuenta, grupo in df.groupby("queried_handle"):
        fechas = grupo["date"].dt.tz_convert("UTC").dt.date.sort_values().tolist()
        if len(fechas) < 2:
            continue
        dias_totales = max((fechas[-1] - fechas[0]).days, 1)
        ritmo = len(fechas) / dias_totales
        for anterior, siguiente in zip(fechas, fechas[1:]):
            dias = (siguiente - anterior).days
            if dias >= dias_minimos and dias * ritmo >= esperadas_minimas:
                filas.append({
                    "cuenta": cuenta, "desde": anterior, "hasta": siguiente,
                    "días": dias, "esperadas": round(dias * ritmo),
                })
    return pd.DataFrame(filas).sort_values("esperadas", ascending=False) if filas else pd.DataFrame()

sospechosos = huecos(df)
print("huecos sospechosos:", len(sospechosos))
sospechosos.head(10)

## 8. Primer análisis

Nada sofisticado: volumen en el tiempo, cuentas más activas y publicaciones con
más favoritos. Suficiente para confirmar que el corpus tiene sentido.

In [ ]:
por_dia = df.set_index("date").resample("D").size()
print("publicaciones por día:")
print(por_dia.to_string())

print()
print("cuentas más activas:")
print(df["queried_handle"].value_counts().head(10).to_string())

print()
print("más favoritos:")
columnas = ["date", "queried_handle", "favorites", "text"]
print(df.nlargest(3, "favorites")[columnas].to_string(index=False, max_colwidth=60))

In [ ]:
# Gráfico opcional: requiere matplotlib
try:
    import matplotlib.pyplot as plt
    eje = por_dia.plot(kind="bar", figsize=(10, 3), title="Publicaciones por día")
    eje.set_xlabel("")
    eje.set_ylabel("publicaciones")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Instala matplotlib para ver el gráfico: pip install matplotlib")

## 9. Pasar a tu propio estudio

1. Copia `censo_plantilla.json` a `censo.json` y llénalo con tus cuentas,
   verificando cada usuario de X contra una fuente institucional. Un usuario
   equivocado no da error: devuelve cero publicaciones.
2. Lanza la descarga completa **fuera del cuaderno**, porque puede durar días:

   ```
   nohup caffeinate -i python -u descargar_cuentas.py --roster censo.json \
       --window-months 3 --retries 12 --retry-backoff 600 >> salida.log 2>&1 &
   ```

   En Windows, con `Start-Process` (sección 6 del README).
3. Mientras corre, puedes volver a este cuaderno y leer el CSV: se reescribe
   completo después de cada lote.
4. Al terminar, corre `reparar_truncamientos.py` y repite las comprobaciones de
   la sección 7 antes de analizar.